# C4 · Produtividade industrial por setor e estado

**Pergunta:** no mesmo setor da indústria, quanto valor cada pessoa ocupada gera por ano em estados diferentes, e quanto pesa a folha nesse valor?

Fonte: IBGE, Pesquisa Industrial Anual (PIA-Empresa), tabela SIDRA 1849, unidades locais industriais de empresas com 5 ou mais pessoas ocupadas, por UF e divisão da CNAE 2.0, 2007 a 2023. Valores em mil reais correntes.

## 1. Carregar e descrever
O arquivo vem da API do SIDRA (`baixar_dados.py`). `X` = valor omitido pelo IBGE por sigilo (poucas empresas na célula); `-` = zero absoluto.

In [1]:
import pandas as pd, numpy as np, json
d = pd.read_csv('dados/pia_1849.csv', dtype={'valor': str})
print(f"{len(d):,} linhas · {d.ano.min()}–{d.ano.max()} · {d.uf.nunique()} UFs · {d.cnae.nunique()} categorias CNAE")
print(round(d.valor.eq('X').mean(), 3), 'das células omitidas por sigilo;', round(d.valor.eq('-').mean(), 3), 'zeradas')

102,816 linhas · 2007–2023 · 27 UFs · 32 categorias CNAE
0.119 das células omitidas por sigilo; 0.082 zeradas


## 2. Montar o cubo ano × setor × estado
Só a indústria de transformação (divisões 10 a 33). Por célula: pessoal ocupado, valor da transformação industrial (VTI), receita líquida e folha (salários + encargos).

**VTI por pessoa** = VTI ÷ pessoal ocupado em 31/12. **Peso da folha** = (salários + encargos) ÷ VTI.

In [2]:
d['v'] = pd.to_numeric(d.valor.replace('-', '0'), errors='coerce')
d['cod'] = d.cnae.str[:2]
t = d[d.cod.str.isdigit() & (d.cod >= '10')]
VARS = {631: 'po', 811: 'vti', 835: 'receita', 673: 'salarios', 834: 'encargos'}
c = (t[t['var'].isin(VARS)].assign(m=lambda x: x['var'].map(VARS))
       .pivot_table(index=['ano', 'cod', 'cnae', 'uf_id', 'uf'], columns='m', values='v').reset_index())
c['folha'] = c.salarios + c.encargos
c['prod'] = c.vti / c.po            # mil R$ por pessoa por ano
c['peso_folha'] = c.folha / c.vti
print(len(c), 'células com algum valor;', c['prod'].notna().sum(), 'com VTI por pessoa')

9980 células com algum valor; 9600 com VTI por pessoa


## 3. Regra de comparação
Célula pequena é instável (uma fábrica muda tudo). Para comparar estados dentro do setor, só entram células com **1.000 pessoas ou mais**, e só setores com **6 estados ou mais** nessa condição no ano. A comparação é sempre dentro do mesmo setor e do mesmo ano, então a inflação não interfere.

In [3]:
MIN_PO, MIN_UF = 1000, 6
def quartis(ano):
    q = c[(c.ano == ano) & (c.po >= MIN_PO) & c['prod'].notna()]
    g = q.groupby(['cod', 'cnae']).agg(n=('uf', 'size'), po=('po', 'sum'),
        p25=('prod', lambda s: s.quantile(.25)), p50=('prod', 'median'), p75=('prod', lambda s: s.quantile(.75)),
        folha=('peso_folha', 'median'))
    g = g[g.n >= MIN_UF].copy(); g['gap'] = g.p75 - g.p25; g['razao'] = g.p75 / g.p25
    return g
g23 = quartis(2023)
print(g23[['n', 'p25', 'p50', 'p75', 'gap', 'razao']].round(1).sort_values('gap').droplevel(1).to_string())

      n    p25    p50     p75     gap  razao
cod                                         
31   20   66.1   86.2   101.4    35.3    1.5
27   14  192.3  218.6   230.7    38.3    1.2
13   16   72.4   98.5   115.2    42.7    1.6
18   14  104.2  122.6   149.7    45.5    1.4
14   20   47.2   59.6    93.4    46.2    2.0
33   19   95.1  106.6   141.3    46.2    1.5
15   16   81.5  102.9   129.1    47.7    1.6
16   16   97.8  127.8   150.0    52.2    1.5
32   13   99.4  120.4   157.6    58.1    1.6
25   22  104.6  144.5   164.5    60.0    1.6
23   24  128.2  171.8   194.1    66.0    1.5
22   19  118.0  154.8   194.3    76.3    1.6
10   26  129.6  159.0   223.9    94.3    1.7
24   16  362.3  415.7   483.6   121.3    1.3
21   12  206.1  286.6   364.4   158.3    1.8
29   13  162.9  214.7   331.7   168.9    2.0
28   13  146.2  277.1   321.8   175.6    2.2
11   23  163.2  257.9   355.1   191.9    2.2
26    9  127.6  152.1   322.4   194.8    2.5
30    8  161.9  247.1   366.5   204.5    2.3
20   21  2

## 4. O número
Para cada setor, a distância entre o estado do quartil de cima e o do quartil de baixo, em VTI por pessoa. O número é a mediana dessa distância entre os setores, multiplicada por 100 pessoas.

In [4]:
gap = g23.gap.median(); raz = g23.razao.median()
print(f"2023 · {len(g23)} setores · {int(g23.n.sum())} combinações setor × estado")
print(f"No mesmo setor, 100 pessoas no quartil de cima dos estados geram R$ {gap*100/1000:.1f} mi a mais de VTI por ano que no quartil de baixo")
print(f"(mediana entre setores; o quartil de cima gera {raz:.2f} vezes o de baixo)")

2023 · 23 setores · 384 combinações setor × estado
No mesmo setor, 100 pessoas no quartil de cima dos estados geram R$ 7.6 mi a mais de VTI por ano que no quartil de baixo
(mediana entre setores; o quartil de cima gera 1.65 vezes o de baixo)


## 5. Peso da folha
Quanto do VTI vai para salários e encargos, somando as células divulgadas, por ano. É razão, então também não depende de inflação.

In [5]:
br = c.groupby('ano')[['vti', 'folha', 'po']].sum()
br['peso_folha'] = br.folha / br.vti
print(br.peso_folha.round(3).to_string())
f23 = br.loc[2023, 'peso_folha']
print(f"2023: R$ {f23*100:.0f} de salários e encargos a cada R$ 100 de VTI")

ano
2007    0.321
2008    0.314
2009    0.352
2010    0.340
2011    0.366
2012    0.378
2013    0.375
2014    0.396
2015    0.400
2016    0.411
2017    0.399
2018    0.366
2019    0.366
2020    0.334
2021    0.286
2022    0.280
2023    0.317
2023: R$ 32 de salários e encargos a cada R$ 100 de VTI


## 6. A dispersão mudou com o tempo?
Mesma regra, ano a ano: mediana da razão quartil de cima ÷ quartil de baixo.

In [6]:
ev = pd.DataFrame({a: {'setores': len(quartis(a)), 'razao': quartis(a).razao.median()} for a in sorted(c.ano.unique())}).T
print(ev.round(2).to_string())

      setores  razao
2007     23.0   1.90
2008     23.0   1.92
2009     23.0   1.75
2010     24.0   1.69
2011     23.0   1.78
2012     24.0   1.69
2013     23.0   1.75
2014     23.0   1.69
2015     24.0   1.70
2016     24.0   1.68
2017     23.0   1.60
2018     23.0   1.60
2019     23.0   1.69
2020     23.0   1.77
2021     23.0   1.71
2022     23.0   1.76
2023     23.0   1.65


## 7. Exportar para o painel
O painel recebe as células brutas e recalcula tudo no navegador, com a mesma regra (1.000 pessoas, 6 estados).

In [7]:
SIG = {11:'RO',12:'AC',13:'AM',14:'RR',15:'PA',16:'AP',17:'TO',21:'MA',22:'PI',23:'CE',24:'RN',25:'PB',26:'PE',27:'AL',28:'SE',29:'BA',
       31:'MG',32:'ES',33:'RJ',35:'SP',41:'PR',42:'SC',43:'RS',50:'MS',51:'MT',52:'GO',53:'DF'}
CURTO = {'10':'Alimentos','11':'Bebidas','12':'Fumo','13':'Têxteis','14':'Vestuário','15':'Couro e calçados','16':'Madeira','17':'Celulose e papel',
 '18':'Impressão','19':'Petróleo e biocombustíveis','20':'Químicos','21':'Farmacêuticos','22':'Borracha e plástico','23':'Minerais não metálicos',
 '24':'Metalurgia','25':'Produtos de metal','26':'Eletrônicos e informática','27':'Materiais elétricos','28':'Máquinas e equipamentos',
 '29':'Veículos','30':'Outros eq. de transporte','31':'Móveis','32':'Produtos diversos','33':'Manutenção e instalação'}
setores = sorted(c.cod.unique()); ufs = sorted(c.uf_id.unique()); anos = sorted(int(a) for a in c.ano.unique())
nomes = c.drop_duplicates('cod').set_index('cod').cnae.str[3:]
def r(x): return None if pd.isna(x) else round(float(x))
cel = [[int(x.ano) - anos[0], setores.index(x.cod), ufs.index(x.uf_id), r(x.po), r(x.vti), r(x.receita), r(x.folha)] for x in c.itertuples()]
dados = {'fonte': 'IBGE · PIA-Empresa · SIDRA 1849', 'unidade': 'mil R$ correntes', 'anos': anos,
         'setores': [{'cod': s, 'curto': CURTO[s], 'nome': nomes[s]} for s in setores],
         'ufs': [{'sigla': SIG[int(u)], 'nome': c[c.uf_id == u].uf.iloc[0]} for u in ufs],
         'campos': ['ano', 'setor', 'uf', 'po', 'vti', 'receita', 'folha'], 'celulas': cel,
         'regra': {'min_po': MIN_PO, 'min_uf': MIN_UF}}
s = 'window.DADOS = ' + json.dumps(dados, ensure_ascii=False, separators=(',', ':')) + ';'
open('painel/dados.js', 'w', encoding='utf-8').write(s)
print(len(cel), 'células ·', round(len(s.encode()) / 1024), 'KB')

9980 células · 336 KB


## 8. Limitações
- **VTI não é lucro nem valor adicionado das contas nacionais.** É o valor bruto da produção menos o custo das operações industriais; não desconta depreciação, despesas comerciais e financeiras.
- **Pessoal ocupado em 31/12**, não horas trabalhadas. Terceirizado não entra; setor que terceiriza muito parece mais produtivo.
- **A divisão da CNAE é ampla.** "Alimentos" junta frigorífico e laticínio; parte da distância entre estados é mistura de produto, não eficiência. A tabela por grupo existe só para Sul e Sudeste.
- **Unidade local, não empresa.** Preço de transferência entre filiais pode deslocar valor entre estados.
- **Células omitidas por sigilo** ficam vazias (não estimadas), o que tira da conta os estados com poucas fábricas no setor.
- **Valores nominais.** Comparações entre anos usam só razões (quartil ÷ quartil, folha ÷ VTI), que não dependem de inflação; o valor em reais é sempre do ano escolhido.
- **Correlação, não causa.** Estar no quartil de baixo indica onde perguntar (automação, escala, mix, preço), não quanto dá para ganhar.